In [1]:
from pathlib import Path
import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [3]:
train_transform = A.Compose([

    A.Resize(224,224),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.Rotate(limit=15, p=0.5),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

val_transform = A.Compose([

    A.Resize(224,224),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

In [4]:
class CattleDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        self.image_paths = []
        self.labels = []

        self.class_names = sorted([
            folder.name
            for folder in self.root_dir.iterdir()
            if folder.is_dir()
        ])

        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(self.class_names)
        }

        for cls_name in self.class_names:

            cls_folder = self.root_dir / cls_name

            for img_path in cls_folder.glob("*"):

                self.image_paths.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls_name]
                )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

In [5]:
train_dataset = CattleDataset(
    root_dir="../datasets/identity_split/train",
    transform=train_transform
)

val_dataset = CattleDataset(
    root_dir="../datasets/identity_split/val",
    transform=val_transform
)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [7]:
class CattleIdentificationModel(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        in_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Sequential(

            nn.Linear(in_features, 1024),

            nn.BatchNorm1d(1024),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(1024, 512),

            nn.ReLU(),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        return self.backbone(x)

In [8]:
num_classes = len(train_dataset.class_names)

model = CattleIdentificationModel(
    num_classes=num_classes
)

model = model.to(device)

print("Classes:", num_classes)

Classes: 84


In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [11]:
model.load_state_dict(
    torch.load("../models/cattle_resnet50.pth")
)

print("Model loaded.")

C:\Users\agarw\AppData\Local\Temp\ipykernel_14816\817360059.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("../models/cattle_resnet50.pth")


Model loaded.


In [12]:
EPOCHS = 25

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss:.4f}")

KeyboardInterrupt: 

In [ ]:
torch.save(
    model.state_dict(),
    "../models/cattle_resnet50.pth"
)

print("Model saved.")

In [13]:
class TrainedEmbeddingModel(nn.Module):

    def __init__(self, trained_model):

        super().__init__()

        self.features = nn.Sequential(
            *list(trained_model.backbone.children())[:-1]
        )

    def forward(self, x):

        x = self.features(x)

        x = x.view(x.size(0), -1)

        return x

In [14]:
trained_embedding_model = TrainedEmbeddingModel(model)

trained_embedding_model = trained_embedding_model.to(device)

trained_embedding_model.eval()

TrainedEmbeddingModel(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (

In [15]:
embedding_database = []
label_database = []

with torch.no_grad():

    for images, labels in train_loader:

        images = images.to(device)

        embeddings = trained_embedding_model(images)

        embedding_database.append(
            embeddings.cpu()
        )

        label_database.append(
            labels
        )

embedding_database = torch.cat(
    embedding_database
)

label_database = torch.cat(
    label_database
)

print(embedding_database.shape)
print(label_database.shape)

torch.Size([838, 2048])
torch.Size([838])


In [16]:
def identify_cow(query_embedding,
                 embedding_database,
                 label_database):

    similarities = F.cosine_similarity(
        query_embedding.unsqueeze(0),
        embedding_database
    )

    best_match_idx = torch.argmax(similarities)

    best_similarity = similarities[
        best_match_idx
    ].item()

    predicted_label = label_database[
        best_match_idx
    ].item()

    return predicted_label, best_similarity

In [17]:
query_image, query_label = train_dataset[50]

query_tensor = query_image.unsqueeze(0).to(device)

with torch.no_grad():

    query_embedding = trained_embedding_model(
        query_tensor
    )

query_embedding = query_embedding.cpu()

predicted_label, similarity = identify_cow(
    query_embedding.squeeze(0),
    embedding_database,
    label_database
)

print("Actual Label:", query_label)
print("Predicted Label:", predicted_label)
print("Similarity:", similarity)

Actual Label: 5
Predicted Label: 5
Similarity: 0.9447273015975952


In [18]:
torch.save(
    model.state_dict(),
    "../models/cattle_resnet50.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [19]:
from pathlib import Path

print(list(Path("../models").glob("*")))

[WindowsPath('../models/cattle_resnet50.pth'), WindowsPath('../models/cow_detector'), WindowsPath('../models/embedding_model'), WindowsPath('../models/muzzle_detector')]


In [20]:
torch.load(
    "../models/cattle_resnet50.pth",
    weights_only=True
)

OrderedDict([('backbone.conv1.weight',
              tensor([[[[-9.1026e-03, -4.3327e-03,  3.5784e-02,  ...,  4.6661e-02,
                         -2.0432e-02,  8.5655e-03],
                        [-5.8031e-02,  4.5005e-02,  7.7703e-02,  ...,  8.8889e-02,
                          3.0421e-02, -5.7480e-02],
                        [ 6.7121e-02, -2.7017e-01,  4.0290e-01,  ..., -1.6450e-01,
                          2.1921e-01, -7.1895e-02],
                        ...,
                        [-1.0888e-01,  3.8105e-01, -4.5408e-01,  ...,  6.8400e-01,
                         -5.7627e-01,  2.2519e-01],
                        [ 2.4659e-02, -1.7724e-01,  6.4298e-01,  ...,  5.2594e-01,
                         -4.8559e-02, -6.7482e-02],
                        [ 4.3714e-02, -1.3139e-01,  1.7097e-02,  ..., -3.5721e-01,
                          1.8984e-01, -2.1607e-02]],
              
                       [[ 6.1465e-03,  2.9064e-03, -1.7683e-02,  ...,  8.5270e-02,
                       